In [1]:
import pandas as pd
import numpy as np
import os
import re
from bs4 import BeautifulSoup
from collections import defaultdict
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import time
from functools import wraps
import math
import json

In [2]:
def export_threads(limit=5000):
    q = pd.read_csv('stackDB/Questions.csv', encoding='latin1', nrows=limit)
    a = pd.read_csv('stackDB/Answers.csv', encoding='latin1')
    t = pd.read_csv('stackDB/Tags.csv', encoding='latin1')

    output_dir = 'documents'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for _, q_row in q.iterrows():
        q_id = q_row['Id']
        
        tags = t[t['Id'] == q_id]['Tag'].fillna('').tolist()
        tags_str = " ".join([str(t) for t in tags])
        
        answers = a[a['ParentId'] == q_id]
        answers_text = ""
        for _, a_row in answers.iterrows():
            answers_text += "\n" + BeautifulSoup(str(a_row['Body']), "html.parser").get_text()
        
        q_body = BeautifulSoup(str(q_row['Body']), "html.parser").get_text()
        
        full_content = f"{q_row['Title']}\n{tags_str}\n{q_body}\n{answers_text}"

        with open(f'{output_dir}/{q_id}.txt', 'w', encoding='utf-8') as f:
            f.write(full_content)

export_threads(5000)

In [12]:
t = pd.read_csv('stackDB/Tags.csv', encoding='latin1')
t.columns

Index(['Id', 'Tag'], dtype='object')

In [16]:
t['Tag'].unique().tolist()

['flex',
 'actionscript-3',
 'air',
 'svn',
 'tortoisesvn',
 'branch',
 'branching-and-merging',
 'sql',
 'asp.net',
 'sitemap',
 'algorithm',
 'language-agnostic',
 'colors',
 'color-space',
 'c#',
 '.net',
 'scripting',
 'compiler-construction',
 'c++',
 'oop',
 'class',
 'nested-class',
 'web-services',
 'sql-server',
 'sql-server-2005',
 'deployment',
 'release-management',
 'visual-studio',
 'versioning',
 'windows',
 'registry',
 'installation',
 'database',
 'loops',
 'connection',
 'file-locking',
 'unix',
 'size',
 'msbuild',
 'cruisecontrol.net',
 'web-applications',
 'dns',
 'subdomain',
 'account',
 '.net-3.5',
 'nant',
 'windows-server-2008',
 'sql-server-2008',
 'unit-testing',
 'testing',
 'version-control',
 'postgresql',
 'stored-procedures',
 'triggers',
 'dataset',
 'datatable',
 'asp-classic',
 'vbscript',
 'html',
 'autocomplete',
 'c',
 'architecture',
 'data-structures',
 'flash',
 'video',
 'embed',
 'powershell',
 'cmdlets',
 'optimization',
 'setter',
 'getter

In [4]:
def benchmark(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        duration = end_time - start_time
        print(f'The method {func.__name__} lasted {duration:.3f} seconds')
        return result
    return wrapper

In [5]:
class InvertedIndexer:
    def __init__(self, doc_dir):
        self.doc_dir = doc_dir
        self.stemmer = PorterStemmer()
        self.stop_words = set(stopwords.words('english'))
        self.index = defaultdict(dict)
        self.doc_list = []

    def preprocess(self, text):
        tokens = re.findall(r'[a-z][a-z#+]*', text.lower())

        return [self.stemmer.stem(t) for t in tokens if t not in self.stop_words and len(t) > 1]
    
    @benchmark
    def build(self):
        self.doc_list = [f for f in os.listdir(self.doc_dir) if f.endswith('.txt')]
        N = len(self.doc_list) # number of docs

        term_counts = defaultdict(lambda: defaultdict(int))

        for doc_name in self.doc_list:
            path = os.path.join(self.doc_dir, doc_name)
            with open(path, 'r', encoding='utf-8') as f:
                content = f.read()
                terms = self.preprocess(content)
                for t in terms:
                    term_counts[t][doc_name] += 1

        for term, docs_with_terms in term_counts.items():
            n_t = len(docs_with_terms) # num of docs where term is
            idf = math.log10(N / n_t)

            for doc_name, count in docs_with_terms.items():

                tf_weight = 1 + math.log10(count)
                self.index[term][doc_name] = tf_weight * idf


    def save(self, filename='index.json'):
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(self.index, f)            

In [6]:
engine = InvertedIndexer('documents')
engine.build()
engine.save()

The method build lasted 64.255 seconds


In [7]:
class BooleanSearch:
    def __init__(self, index_path = 'index.json'):
        with open(index_path, 'r', encoding = 'utf-8') as f:
            self.index = json.load(f)
        self.stemmer = PorterStemmer()
        self.all_docs = set()

        for doc_dict in self.index.values():
            self.all_docs.update(doc_dict.keys())  

    def get_postfix(self, query):
        priority = {'NOT': 3, 'AND': 2, 'OR': 1, '(': 0}
        stack, output = [], []
        
        tokens = re.findall(r'\(|\)|AND|OR|NOT|[a-zA-Z#+]+', query, re.IGNORECASE)
        
        for token in tokens:
            t_upper = token.upper()
            if t_upper == '(': 
                stack.append(token)
            elif t_upper == ')':
                while stack and stack[-1] != '(': 
                    output.append(stack.pop())
                if stack: stack.pop()
            elif t_upper in priority:
                # Priorita operátorů
                while stack and priority.get(stack[-1].upper(), 0) >= priority[t_upper]:
                    output.append(stack.pop())
                stack.append(t_upper)
            else:
                stemmed = self.stemmer.stem(token.lower())
                output.append(stemmed)
        
        while stack: 
            output.append(stack.pop())
        return output
    @benchmark
    def search(self, query, p=2):
        postfix = self.get_postfix(query)
        results = {}

        for doc_id in self.all_docs:
            stack = []
            for token in postfix:
                if token == 'AND':
                    w2 = stack.pop() if stack else 0
                    w1 = stack.pop() if stack else 0
                    score = 1 - ((((1-w1)**p + (1-w2)**p) / 2)**(1/p))
                    stack.append(score)
                elif token == 'OR':
                    w2 = stack.pop() if stack else 0
                    w1 = stack.pop() if stack else 0
                    score = (((w1**p + w2**p) / 2)**(1/p))
                    stack.append(score)
                elif token == 'NOT':
                    w1 = stack.pop() if stack else 0
                    stack.append(1 - w1)
                else:
                    val = self.index.get(token, {}).get(doc_id, 0)
                    stack.append(val)
            
            if stack:
                final_relevance = stack.pop()
                if final_relevance > 0.05: 
                    results[doc_id] = round(final_relevance, 4)

        return sorted(results.items(), key=lambda x: x[1], reverse=True)

In [8]:
@benchmark
def sequential_search(doc_dir, term):
    results = []

    term_stemmed = PorterStemmer().stem(term.lower())

    for filename in os.listdir(doc_dir):
        if filename.endswith('.txt'):
            with open(os.path.join(doc_dir, filename), 'r', encoding='utf-8') as f:
                if term_stemmed in f.read().lower():
                    results.append(filename)
    return results                

In [17]:
searcher = BooleanSearch('index.json')
res_idx = searcher.search("jupyter")

res_seq = sequential_search('documents', 'jupyter')

The method search lasted 0.001 seconds
The method sequential_search lasted 0.682 seconds


In [ ]:
searcher = BooleanSearch('index.json')

results = searcher.search("jupyter")
print(results[:10])

The method search lasted 0.001 seconds
[]


: 